# Import Thư viện



In [ ]:
import requests
import json
import csv
import time
import uuid
import sys
import os

# File cần chạy


In [ ]:
# API của chatbot (đảm bảo backend chạy)
API_URL = "http://127.0.0.1:5000/api/chat"
INPUT_FILE = "../questions.txt"
OUTPUT_FILE = "../report_ket_qua.csv"

# Rotate API Key

In [ ]:
# --- DANH SÁCH KEY (QUEUE) ---
# Bạn hãy điền tất cả key của bạn vào đây. 
# Code sẽ chạy lần lượt từ trên xuống dưới.
GEMINI_KEYS = [
    "AIzaSy...Key1",
    "AIzaSy...Key2",
    "AIzaSy...Key3",
    # Dán thêm bao nhiêu key tùy thích
]

# Xử lý câu hỏi + câu trả lời

In [ ]:
def ask_chat(question, thread_id,api_key):
    headers={
        'Content-Type': 'application/json',
        'x-gemini-api-key': api_key
             }
    payload = {
        "message": question,
        "thread_id": thread_id }
    full_answer=""
    process_steps=[] 
    start_time=time.time()
     # Gọi API với stream=True để bắt từng event
    response=requests.post(API_URL, json=payload, headers=headers, stream=True)
    if response.status_code != 200:
            return f"Error {response.status_code}", [], 0
    
    for line in response.iter_lines():
        if line:
            decoded_line=line.decode('utf-8')
            #bỏ qua tiêu đề thừa, chỉ xử lý data:
            if decoded_line.startswith("event:log"):
                continue
            if decoded_line.startswith("event: message"):
                        continue
            if decoded_line.startswith("data:"):
                json_str = decoded_line.replace("data: ", "")
                data = json.loads(json_str)
                if "step" in data:
                    step_name = data.get("step")
                                # Chỉ lấy các bước quan trọng
                    if step_name in ["call_rag_agent", "call_sql_agent", "generate_response"]:
                        process_steps.append(step_name)
                if "token" in data:
                    full_answer += data["token"]
                if "error" in data:
                    full_answer = f"LỖI TỪ BOT: {data['error']}"
    duration = round(time.time() - start_time, 2)
    return full_answer.strip(), process_steps, duration

# Run + ghi vào file kết quả

In [ ]:
print("--- BẮT ĐẦU CHẠY AUTO TEST ---")

# Kiểm tra file câu hỏi
if not os.path.exists(INPUT_FILE):
    print(f"Lỗi: Không tìm thấy file {INPUT_FILE}")

# Kiểm tra danh sách Key
num_keys=len(GEMINI_KEYS)
if num_keys == 0:
    print("Lỗi: Bạn chưa điền Key vào danh sách GEMINI_KEYS trong code!")

# Mở file ghi kết quả (CSV)
with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8-sig') as csv_file:
    fieldnames = ['STT', 'Câu hỏi', 'Câu trả lời của Bot', 'Quy trình xử lý', 'Thời gian (s)']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    # Đọc danh sách câu hỏi
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        questions = [line.strip() for line in f if line.strip()]
    total = len(questions)
    print(f"Đã tìm thấy {total} câu hỏi. Đang xử lý...\n")
    for idx, question in enumerate(questions):
        num_keys = len(GEMINI_KEYS)
        if idx > 0 and idx % num_keys == 0:
                print(f"Wait 4s...")
                time.sleep(4)
        current_key = GEMINI_KEYS[idx % num_keys]
        # Tạo thread_id ngẫu nhiên cho mỗi câu hỏi để không bị lẫn context cũ
        thread_id = str(uuid.uuid4())
        
        print(f"[{idx}/{total}] Đang hỏi: {question} ...", end="\r")
        
        # Gửi câu hỏi
        answer, steps, duration = ask_chat(question, thread_id,current_key)
        
        # Format lại quy trình cho đẹp
        steps_str = " -> ".join(steps) if steps else "Direct/General"

        # Ghi vào file CSV
        writer.writerow({
            'STT': idx,
            'Câu hỏi': question,
            'Câu trả lời của Bot': answer,
            'Quy trình xử lý': steps_str,
            'Thời gian (s)': duration
        })
        
        print(f"[{idx}/{total}] Xong! ({duration}s) - {steps_str}")

print(f"\n--- HOÀN THÀNH! ---")
print(f"Kết quả đã được lưu tại: {OUTPUT_FILE}")